In [ ]:
%%capture
pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import os
import time
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Блок констант
main_url = "https://www.kp.ru/russia/krym/dostoprimechatelnosti/"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
pattern_title = r'^\d+\.\s(.+)'
pattern_title_eng = r'/([^/]+)/$'

In [ ]:
# Отправляем GET‑запрос
response = requests.get(main_url, headers=headers)
response.encoding = 'utf-8'  # Указываем кодировку

# Парсим HTML
soup = BeautifulSoup(response.text, 'html.parser')

# Смотрим, что там пришло
for item in soup.find_all('body'):
    print(item)

<body>
<div>
<!--AdFox START-->
<!--kp_sites-->
<!--Площадка: kp.ru - Отдых в России / * / *-->
<!--Тип баннера: Rest (mobile, tablet) - fullscreen-->
<!--Расположение: верх страницы-->
<div id="adfox_169168148266335981"></div>
<script>
    window.yaContextCb.push(()=>{
        Ya.adfoxCode.createAdaptive({
            ownerId: 232598,
            containerId: 'adfox_169168148266335981',
            params: {
                pp: 'g',
                ps: 'dars',
                p2: 'imse',
                puid1: '',
                puid2: '',
                puid3: '',
                puid4: '',
                puid5: '',
                puid6: '',
                puid7: ''
            }
        }, ['tablet', 'phone'], {
            tabletWidth: 1023,
            phoneWidth: 639,
            isAutoReloads: false
        })
    })
</script> </div>
<!-- Google Tag Manager (noscript) -->
<noscript><iframe height="0" src="https://www.googletagmanager.com/ns.html?id=GTM-TD34GL5" style="displ

In [ ]:
# Проверяем, что в описании 1 достопримечательности
url_dost = "https://www.kp.ru/russia/yalta/mesta/dvorets-zamok-lastochkino-gnezdo/"

response = requests.get(url_dost, headers=headers)
response.encoding = 'utf-8'  # Указываем кодировку

# Парсим HTML
soup = BeautifulSoup(response.text, 'html.parser')

for item in soup.find_all('body'):
    print(item)

<body>
<div>
<!--AdFox START-->
<!--kp_sites-->
<!--Площадка: kp.ru - Отдых в России / * / *-->
<!--Тип баннера: Rest (mobile, tablet) - fullscreen-->
<!--Расположение: верх страницы-->
<div id="adfox_169168148266335981"></div>
<script>
    window.yaContextCb.push(()=>{
        Ya.adfoxCode.createAdaptive({
            ownerId: 232598,
            containerId: 'adfox_169168148266335981',
            params: {
                pp: 'g',
                ps: 'dars',
                p2: 'imse',
                puid1: '',
                puid2: '',
                puid3: '',
                puid4: '',
                puid5: '',
                puid6: '',
                puid7: ''
            }
        }, ['tablet', 'phone'], {
            tabletWidth: 1023,
            phoneWidth: 639,
            isAutoReloads: false
        })
    })
</script> </div>
<!-- Google Tag Manager (noscript) -->
<noscript><iframe height="0" src="https://www.googletagmanager.com/ns.html?id=GTM-TD34GL5" style="displ

In [ ]:
# Создаемм необходимые папки
os.makedirs("Krym", exist_ok=True)
os.makedirs("Krym/Pictures", exist_ok=True)
os.makedirs("Krym/Description", exist_ok=True)

In [ ]:
# Общий файл
sights = []

response = requests.get(main_url, headers=headers)
response.encoding = 'utf-8'  # Указываем кодировку

soup = BeautifulSoup(response.text, 'html.parser')

for item in soup.find_all('div', class_= 'full-card'):
    title = item.find('div', class_= 'full-card-content').find('h3').find('a').text.strip()
    if re.match(pattern_title, title):
        sights.append({'url': item.find('div', class_ = 'full-card-image').find('a')['href'], 'title': re.search(pattern_title, title).group(1)})

ul = soup.find('ul', class_= 'wp-block-list')
for li in ul.find_all('li'):
  a = li.find('a')
  sights.append({'url': a['href'], 'title': a.text.strip()})

with open('Krym/common.txt', 'w') as file:
    for item in sights:
        file.write(item.get('title')+'\n')
        file.write(re.search(pattern_title_eng, item.get('url')).group(1)+'\n')
        file.write(item.get('url')+'\n')
        file.write('\n')

In [ ]:
# Функция парсинга одной достопримечательности
def parse_sight_info (url, headers=headers, enc='utf-8'):
  title_eng = re.search(pattern_title_eng, url).group(1)
  response = requests.get(url, headers)
  response.encoding = enc  # Указываем кодировку

  text = []

  # Парсим HTML
  soup = BeautifulSoup(response.text, 'html.parser')

  img_url = soup.find('img', class_ = "img-hide-on-mobile")['src']
  !wget '{img_url}' -O 'Krym/Pictures/{title_eng}.jpg'

  for p in soup.find_all('p', {'dir': 'ltr'}):
    text.append(p.text.strip())
  text.append('')

  for section in soup.find_all('section', class_ = 'wp-block-rest-gutenberg-blocks-guten-block content-block'):
    div = section.find('div', class_ = 'text-block')
    for h in section.find_all (['h2', 'h3']):
      if h.text.strip() in ['История', 'Что посмотреть'] or h.text.strip().endswith('в разное время года') or (h.text.strip().startswith('Где сделать') and h.text.strip().endswith('фото')):
        text.append (h.text.strip())
        for p in div.find_all(['p']):
          text.append(p.text.strip())
        text.append('')
      elif h.text.strip() in ['Интересные факты']:
        text.append (h.text.strip())
        for li in div.find_all(['li']):
          text.append(li.text.strip())
        text.append('')
      elif h.text.strip() in ['Цены']:
        text.append (h.text.strip())
        for td in div.select('.wp-block-table td'):
          text.append(td.text.strip())
        text.append('')
      elif h.text.strip() in ['Как добраться']:
        text.append (h.text.strip())
        for div2 in div.find_all ('div', class_ = 'route-description'):
          text.append(div2.text.strip())
        text.append('')
      else:
        continue

  with open(f'Krym/Description/{title_eng}.txt', 'w', encoding='utf-8') as file:
    file.write('\n'.join(text))

In [ ]:
# Парсим все достопримечательности
for sight in sights:
  parse_sight_info (sight.get('url'))
  time.sleep(8)

--2026-04-21 07:10:41--  https://s8.stc.all.kpcdn.net/russia/wp-content/uploads/2019/05/gnezdo-1330-530x322.jpg
Resolving s8.stc.all.kpcdn.net (s8.stc.all.kpcdn.net)... 188.72.103.110
Connecting to s8.stc.all.kpcdn.net (s8.stc.all.kpcdn.net)|188.72.103.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 182541 (178K) [image/jpeg]
Saving to: ‘Krym/Pictures/dvorets-zamok-lastochkino-gnezdo.jpg’

Krym/Pictures/dvore 100%[===================>] 178.26K   399KB/s    in 0.4s    

2026-04-21 07:10:43 (399 KB/s) - ‘Krym/Pictures/dvorets-zamok-lastochkino-gnezdo.jpg’ saved [182541/182541]

--2026-04-21 07:10:53--  https://s4.stc.all.kpcdn.net/russia/wp-content/uploads/2024/08/livadijskij-dvorets-v-yalte-1330-530x322.jpg
Resolving s4.stc.all.kpcdn.net (s4.stc.all.kpcdn.net)... 188.72.103.110
Connecting to s4.stc.all.kpcdn.net (s4.stc.all.kpcdn.net)|188.72.103.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 175656 (172K) [image/jpeg]
Saving t

In [ ]:
# Забираем файлы на свой диск
!cp -r /content/Krym /content/drive/MyDrive/Krym

In [ ]:
keys = []
for i in range(100): #sights:
  if sights[i].get('url') == 'https://www.kp.ru/russia/sudak/mesta/genuezskaya-krepost/':
    print (i, sights[i].get('title'))

13 Генуэзская крепость
38 Генуэзская крепость в Феодосии


После загрузки оказалось файлов в описании и картинках не по 100, а по 99 штук. После анализа, я нашла две одинаковые ссылки. Таким образом, реально всего достопримечательностей 99, но, видимо, чтобы добить до красивого числа 100, одну достопримечательность отобразили два раза. Но меня не проведешь) Также одной картинки не было (Национальный парк Крымский). Ее реально нет в описании на нужном месте, добавила вручную другую картинку.
В некоторых описаниях отсутствуют куски текста, так как описания достопримечательностей размечены не идентично. Так как проект учебный, я пока оставила так. По моему мнению датасет достсточный, есть с чем работать. В реальном проекте, конечно, надо просматривать все описания и дописывать парсер или добавлять вручную.